# PandasSearchの使い方

### PandasSearchクラスのインポート

In [1]:
from pandas_search.pandas_search import PandasSearch
import pandas as pd

### サンプルデータの作成

In [2]:
df = pd.DataFrame([["taro", 23, "male", "japan"],
                   ["rin", 18, "male", "japan"],
                   ["hana", 33, "female", "japan"],
                   ["mike", 40, "male", "usa"],],
                   columns=["name", "age", "sex", "country"])
print(df)   # 確認

   name  age     sex country
0  taro   23    male   japan
1   rin   18    male   japan
2  hana   33  female   japan
3  mike   40    male     usa


### 簡単な使い方の例

In [3]:
ps = PandasSearch(df)
searched_cells = ps.search("hana")
target = ps.peek(searched_cells, target_size=(1, -1))
target


[   name  age     sex country
 2  hana   33  female   japan]

上のコードでは、データフレームdf内に含まれる文字列「hana」を探索して、  
「hana」の文字列から、その行の行末までのデータを抽出しています  
抽出結果targetは、データフレームからなるリスト形式で返されます

詳細は以降で説明していきます。

### 関数searchの使い方
関数searchは、データフレーム内に含まれる、対象文字列の位置をGeneratorオブジェクトとして返します

In [4]:
ps = PandasSearch(df)
searched_cells = ps.search("hana")

list(searched_cells)

[(2, 0)]

> ps = PandasSearch(df)

データフレームdfを引数として、PandasSearchクラスのインスタンスを作成します

> searched_cells = ps.search("hana")

「hana」の文字列をデータフレームdf内で探索します

> list(searched_cells)

search関数は、内部でyieldを用いており、Generatorオブジェクトを返します  
df内に「hana」は2行0列目にしかないので、  
[(2, 0)]の結果が出力されます

In [5]:
search_default = ps.search("male")  # デフォルトの検索
search_not_exact = ps.search("male", exact_match=False) # 部分一致検索
search_exact = ps.search("male", exact_match=True) # 完全一致検索
print(f"default search: {list(search_default)}")
print(f"not exact search: {list(search_not_exact)}")
print(f"exact search: {list(search_exact)}")

default search: [(0, 2), (1, 2), (2, 2), (3, 2)]
not exact search: [(0, 2), (1, 2), (2, 2), (3, 2)]
exact search: [(0, 2), (1, 2), (3, 2)]


デフォルトでは、部分検索で探索します  
オプションの「exact_match=False」がデフォルトなので、同じ結果が返っています  
exact_matchをTrueに設定すると、完全一致検索の結果が返ってきます

In [6]:
searched_cells = ps.search("^male$")
list(searched_cells)   # 正規表現で検索

[(0, 2), (1, 2), (3, 2)]

対象文字列は、正規表現で記述してデータフレーム内の位置を検索することもできます

In [7]:
searched_cells = ps.search("3")
list(searched_cells)

[(0, 1), (2, 1)]

ここでは、文字列「3」を含む位置を部分一致で検索しています  
データフレーム内で3は、年齢（age）の列の23（0行1列目）と33（2行1列目）にあらわれます  
そのため、結果は[(0, 1), (2, 1)]になります


### 関数peekの使い方
関数peekは、search関数で求めた対象文字列の位置データに基づいて、データフレームdfからデータを抽出します

In [8]:
searched_cells = ps.search("3")
ps.peek(searched_cells)

[   age
 0   23,
    age
 2   33]

デフォルトでは、search関数で探索した位置情報自体にあるデータを返します  
ここでは、age列の0行目と2行目に対応する「23」と「33」を返しています


In [9]:
searched_cells = ps.search("3")
ps.peek(searched_cells, target_size=(1, 1))

[   age
 0   23,
    age
 2   33]

デフォルトと同じ、target_size=(1, 1)に設定すると、当然、同じ結果が返ってきます。

In [10]:
searched_cells = ps.search("3")
ps.peek(searched_cells, target_size=(2, 2))

[   age   sex
 0   23  male
 1   18  male,
    age     sex
 2   33  female
 3   40    male]

taget_sizeに(2, 2)を設定した場合、searched_cellsの位置から2行2列のマトリクスを抽出します。

In [11]:
searched_cells = ps.search("3")
ps.peek(searched_cells, target_size=(2, -1)) # -1は「最後まで」を意味

[   age   sex country
 0   23  male   japan
 1   18  male   japan,
    age     sex country
 2   33  female   japan
 3   40    male     usa]

target_sizeを (2, -1)に設定します。このとき、列指定の-1は「列の最後まで」を意味します  
このため、age列から、最後のcountry列までが抽出されています

In [12]:
searched_cells = ps.search("3")
ps.peek(searched_cells, target_size=(2, -2)) # -2は「最後から2つ前まで」を意味

[   age   sex
 0   23  male
 1   18  male,
    age     sex
 2   33  female
 3   40    male]

今度はtarget_sizeに、(2, -2) を設定しました。ここで、列の指定 -2 は、最後から2つ前の列を意味します  
このため、countryの前の列 sex までが抽出されます

In [13]:
searched_cells = ps.search("3")
ps.peek(searched_cells, target_size=(-1, 2))

[   age     sex
 0   23    male
 1   18    male
 2   33  female
 3   40    male,
    age     sex
 2   33  female
 3   40    male]

行でも同じです。target_sizeに (-1, 2)を設定すると、行末までを抽出します

### 関数 rsearch /csearch

行または列から指定した単語の位置を探索する関数として、rsearchとcsearchを用意しました。  
  
rsearchはrows=[start_row, end_row]で指定した開始行start_rowと最終行end_row内から  
対象となる文字列の位置を探索します

In [14]:
searched_cells = ps.rsearch("3", rows=[1, 3])
ps.peek(searched_cells)



[   age
 2   33]

csearchはcols=[start_cos, end_col]で指定した開始列start_colと最終列end_col内から  
対象となる文字列の位置を探索します

In [33]:
searched_cells = ps.csearch("i", cols=[0, 2])
ps.peek(searched_cells, target_size=(1, 3))


[  name  age   sex
 1  rin   18  male,
    name  age   sex
 3  mike   40  male]